# TWZRD Agent Intel: Trust Verification for AI Agents on Solana

This notebook demonstrates how to use [TWZRD Agent Intel](https://intel.twzrd.xyz) — a remote MCP server that provides trust scoring and x402 payment verification for AI agents on Solana — in combination with the Anthropic Claude API.

**TWZRD Agent Intel** enables Claude-powered agents to:
- Score the trustworthiness of counterparty agents (0-100 trust score)
- Run preflight checks before x402 micropayments
- Verify x402 payment receipts from agent-to-agent transactions

> **Zero install:** TWZRD Agent Intel is a remote MCP server — no local installation needed for basic use.

## Setup

In [ ]:
!pip install anthropic mcp

In [ ]:
import asyncio
import anthropic
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

TWZRD_MCP_URL = "https://intel.twzrd.xyz/mcp"

## Part 1: Direct MCP tool calls

In [ ]:
async def score_agent(wallet: str) -> str:
    """Score a Solana agent wallet using TWZRD Agent Intel."""
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("score_agent", {"wallet": wallet})
            return result.content[0].text

# Example: score an agent wallet
wallet = "D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi"
score = asyncio.run(score_agent(wallet))
print(f"Trust score for {wallet}:"  )
print(score)

## Part 2: Claude agent with TWZRD trust verification

This section shows how to give Claude access to TWZRD tools and let it decide
whether to trust a counterparty agent before proceeding.

In [ ]:
import os

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# Define TWZRD tools for Claude
twzrd_tools = [
    {
        "name": "score_agent",
        "description": "Score a Solana agent wallet. Returns trust score (0-100) and reputation data."",
        "input_schema": {
            "type": "object",
            "properties": {"wallet": {"type": "string", "description": "Solana wallet address"}},
            "required": ["wallet"]
        }
    },
    {
        "name": "preflight_check",
        "description": "Pre-transaction trust check. Returns safe_to_transact boolean."",
        "input_schema": {
            "type": "object",
            "properties": {"wallet": {"type": "string", "description": "Solana wallet address"}},
            "required": ["wallet"]
        }
    }
]

async def execute_tool(tool_name: str, tool_input: dict) -> str:
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, tool_input)
            return result.content[0].text

async def claude_trust_check(agent_wallet: str, task: str) -> str:
    """Ask Claude to verify an agent wallet and decide whether to proceed with a task."""
    messages = [{
        "role": "user",
        "content": f"Before proceeding, check if wallet {agent_wallet} is trustworthy. If trust score >= 50, proceed with: {task}"
    }]
    
    while True:
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            tools=twzrd_tools,
            messages=messages
        )
        
        if response.stop_reason == "end_turn":
            return response.content[0].text
        
        # Execute tool calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = asyncio.run(execute_tool(block.name, block.input))
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

# Run the trust check
result = asyncio.run(claude_trust_check(wallet, "fetch pricing data from this agent"))
print(result)

## MCP Configuration

To add TWZRD Agent Intel to Claude Desktop or other MCP clients:

```json
{"mcpServers": {"twzrd-agent-intel": {"url": "https://intel.twzrd.xyz/mcp"}}}
```

**Tools available (free):**
- `score_agent(wallet)` — trust score + reputation data
- `resolve_agent(wallet)` — agent identity resolution
- `preflight_check(wallet)` — pre-transaction safety check
- `verify_trust_receipt(receipt)` — verify x402 payment receipt

**Paid tool (x402 micropayment):**
- `get_trust_receipt(wallet)` — full trust receipt for agent identity